In [20]:
import numpy as np
import pandas as pd

df_players = pd.read_csv("../data/Joueurs National 1.csv")

cols = ["RunDist", "HiSpeedRunDist", "SprintDist", "MinIncET"]

for col in cols:
    # convert safely to numeric, coerce bad strings like "RunDist" into NaN
    df_players[col] = (
        df_players[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df_players[col] = pd.to_numeric(df_players[col], errors="coerce")

# drop rows where minutes or the metrics are missing
df_players = df_players.dropna(subset=["MinIncET", "RunDist", "HiSpeedRunDist", "SprintDist"])

# drop invalid minutes (0)
df_players = df_players[df_players["MinIncET"] > 0]

# Filter realistic match minutes (avoid season totals / corrupted rows)
df_players = df_players[(df_players["MinIncET"] >= 10) & (df_players["MinIncET"] <= 130)]

# Filter unrealistic match distances (remove aggregated / corrupted rows)
df_players = df_players[
    (df_players["RunDist"] <= 3000) &
    (df_players["HiSpeedRunDist"] <= 1500) &
    (df_players["SprintDist"] <= 600)
]





In [21]:
[c for c in df_players.columns if "HighIntensity" in c]



[]

In [26]:
df_players["HighIntensity15plus"] = (
    df_players["RunDist"]
    + df_players["HiSpeedRunDist"]
    + df_players["SprintDist"]
)

df_players["HighIntensity15plus_per90"] = df_players["HighIntensity15plus"] / df_players["MinIncET"] * 90
df_players.head()


,Rank,playerId,playerImageId,player,playerFullName,pos,teamImageId,teamName,teamShortName,teamAbbrevName,...,finalScoreOpponent,gameStatus,result,scatterId,scatterExtra,optaTeamId,teamId,team,HighIntensity15plus,HighIntensity15plus_per90
3,1,7upcymlpcp1zi8gojnl4krnpx,177277,B. Youssouf,Bendjaloud Youssouf,Right Back,693,Sochaux,Sochaux,SOC,...,0,Played,W 1-0,2585255,PUY vs SOC (2025-08-08),693,d9kwaceoj54pfprd6oc2hwae6,Sochaux,2370.3,2176.806122
4,1,1v77h0qhlk8mxixqpraz77zyt,73847,B. Angoua,Benjamin Angoua,Right Centre Back,3271,Stade Briochin,Stade Briochin,SBR,...,1,Played,L 0-1,2585259,SBR vs VRS (2025-08-08),3271,rv0okxl2zv1ow3me605xl8b5,Stade Briochin,1485.0,1392.187500
5,1,5rk9y1ybsvkgdepofru7u9psl,94347,R. Thomas,Romain Thomas,Left Centre Back,1996,Valenciennes,Valenciennes,VFC,...,0,Played,T 0-0,2585252,CHA vs VFC (2025-08-08),1996,g8cvbv4wrvq7uf4347yjkd35,Valenciennes,1541.8,1415.938776
6,1,1zlc0btscda5bstd4z2hz9zv9,92893,M. Peybernes,Mathieu Peybernes,Defensive Midfielder,693,Sochaux,Sochaux,SOC,...,0,Played,W 1-0,2585255,PUY vs SOC (2025-08-08),693,d9kwaceoj54pfprd6oc2hwae6,Sochaux,319.9,1439.550000
7,1,3pp5knrwnqq5p8m2ovijhladx,124533,P. Delecroix,Paul Delecroix,Goalkeeper,2130,Dijon,Dijon,DIJ,...,1,Played,W 2-1,2585256,ORL vs DIJ (2025-08-08),2130,79dc2mv3x269nhix3mkhc5wfu,Dijon,172.7,161.906250


# --- LIGUE 2 CLEANING ---


In [30]:
df_l2 = pd.read_csv("../data/Joueurs Ligue 2.csv")


In [31]:
cols = ["RunDist", "HiSpeedRunDist", "SprintDist", "MinIncET"]

for col in cols:
    df_l2[col] = (
        df_l2[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    df_l2[col] = pd.to_numeric(df_l2[col], errors="coerce")

df_l2 = df_l2.dropna(subset=cols)
df_l2 = df_l2[df_l2["MinIncET"] > 0]


In [32]:
df_l2 = df_l2[(df_l2["MinIncET"] >= 10) & (df_l2["MinIncET"] <= 130)]


In [33]:
df_l2 = df_l2[
    (df_l2["RunDist"] <= 3000) &
    (df_l2["HiSpeedRunDist"] <= 1500) &
    (df_l2["SprintDist"] <= 600)
]


In [34]:
df_l2["HighIntensity15plus"] = (
    df_l2["RunDist"]
    + df_l2["HiSpeedRunDist"]
    + df_l2["SprintDist"]
)

df_l2["HighIntensity15plus_per90"] = (
    df_l2["HighIntensity15plus"] / df_l2["MinIncET"] * 90
)


In [35]:
df_l2[["RunDist", "HiSpeedRunDist", "SprintDist"]].describe()
df_l2["HighIntensity15plus_per90"].quantile([0.5, 0.9, 0.95, 0.99])


0.50    2243.056719
0.90    3062.666146
0.95    3326.936848
0.99    3908.864455
Name: HighIntensity15plus_per90, dtype: float64

In [36]:
df_l2["competition"] = "Ligue 2"
df_players["competition"] = "National 1"


In [37]:
df_all = pd.concat([df_players, df_l2], ignore_index=True)


In [38]:
df_all["competition"].value_counts()


competition
Ligue 2       4358
National 1    3457
Name: count, dtype: int64

In [39]:
df_all.to_csv("../data/prepped_players_v2.csv", index=False)


In [40]:
# --- Streamlit compatibility columns ---

# 20–25 km/h per90
df_all["HiSpeedRunDist_per90"] = df_all["HiSpeedRunDist"] / df_all["MinIncET"] * 90

# Sprint >25 km/h per90
df_all["SprintDist_per90"] = df_all["SprintDist"] / df_all["MinIncET"] * 90

# Total High Speed per90 (WHAT STREAMLIT EXPECTS)
df_all["TotalHighSpeedDist_per90"] = df_all["HighIntensity15plus_per90"]


In [41]:
df_all[[
    "TotalHighSpeedDist_per90",
    "HiSpeedRunDist_per90",
    "SprintDist_per90"
]].describe()


,TotalHighSpeedDist_per90,HiSpeedRunDist_per90,SprintDist_per90
count,7815.000000,7815.000000,7815.000000
mean,2034.071189,525.240898,171.288234
std,925.821567,266.017257,124.930579
min,0.000000,0.000000,0.000000
25%,1652.968085,387.559961,82.324675
50%,2183.104478,557.444444,154.206186
75%,2622.164835,695.790865,245.512577
max,5184.692308,1697.400000,899.500000


In [42]:
df_all.to_csv("../data/prepped_players_v2.csv", index=False)
